# Regression Modelling

This notebook establishes a baseline, compares regression models, and tunes
promising candidates using cross-validation on training data.

The main population is all 395 Mathematics students, including 38 recorded
zero final grades. The 80/20 split assigns 316 students to training and 79 to
held-out testing. G1 and G2 remain excluded. Test performance is reserved for
Notebook 04; the full dataset was explored in Notebook 01, so it is not a fully
blind exploratory holdout.

Previous positive-final-grade-only results describe a different population
and are archived in `../docs/experiments/positive_grades_only_results.json`.
They should not be compared as if only a model setting had changed.


## Baseline and Validation Summary

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


current_folder = Path.cwd()
PROJECT_ROOT = (
    current_folder
    if (current_folder / "data").is_dir()
    else current_folder.parent
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import make_preprocessor

In [2]:
df = pd.read_csv(
    PROJECT_ROOT / "data" / "student-mat.csv",
    sep=";"
)

# Retain all recorded grades, including zero, as in Notebook 02.
assert df["G3"].notna().all(), "Missing final grades require investigation."
assert df["G3"].between(0, 20).all(), "Final grades must be between 0 and 20."

X = df.drop(columns=["G1", "G2", "G3"])
y = df["G3"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (316, 30) (316,)
Test: (79, 30) (79,)


In [3]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

In [4]:
baseline = DummyRegressor(strategy="mean")

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

In [5]:
baseline_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -baseline_scores["test_MAE"],
    "RMSE": -baseline_scores["test_RMSE"],
    "R2": baseline_scores["test_R2"]
})

baseline_folds

,Fold,MAE,RMSE,R2
0,1,2.723090,4.009308,-0.119523
1,2,4.247757,5.488188,-0.000260
2,3,2.639877,3.748342,-0.000336
3,4,3.554489,4.581837,-0.054337
4,5,3.849112,4.989614,-0.000660


In [6]:
baseline_summary = (
    baseline_folds[["MAE", "RMSE", "R2"]]
    .agg(["mean", "std"])
)

baseline_summary

,MAE,RMSE,R2
mean,3.402865,4.563458,-0.035023
std,0.703600,0.708620,0.052692


## Linear Regression

Linear Regression provides an interpretable first model for assessing
whether linear relationships between student features and final grades
improve upon the mean-prediction baseline.

Numerical and ordinal features are standardized. Original absences
are retained initially. One category per nominal feature is omitted
as a reference category to avoid redundant dummy columns with an intercept.


In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

linear_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=True
)

linear_preprocessor.set_params(
    column_processing__nominal__drop="first"
)

linear_pipeline = Pipeline([
    ("preprocessing", linear_preprocessor),
    ("model", LinearRegression())
])

In [8]:
linear_scores = cross_validate(
    linear_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

linear_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -linear_scores["test_MAE"],
    "RMSE": -linear_scores["test_RMSE"],
    "R2": linear_scores["test_R2"]
})

linear_folds

,Fold,MAE,RMSE,R2
0,1,3.232419,4.484516,-0.400635
1,2,3.912047,4.876418,0.210309
2,3,3.059276,4.033239,-0.158178
3,4,4.030508,4.936385,-0.223822
4,5,3.379134,4.704619,0.110386


In [9]:
metrics = ["MAE", "RMSE", "R2"]

comparison = pd.DataFrame({
    "Mean baseline": baseline_folds[metrics].mean(),
    "Linear Regression": linear_folds[metrics].mean()
}).T

comparison

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Linear Regression,3.522677,4.607035,-0.092388


In [10]:
linear_folds[metrics].agg(["mean", "std"])

,MAE,RMSE,R2
mean,3.522677,4.607035,-0.092388
std,0.426936,0.365655,0.249681


## Ridge Regression

In [11]:
from sklearn.linear_model import Ridge

ridge_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=True
)

ridge_preprocessor.set_params(
    column_processing__nominal__drop="first"
)

ridge_pipeline = Pipeline([
    ("preprocessing", ridge_preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_scores = cross_validate(
    ridge_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

ridge_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -ridge_scores["test_MAE"],
    "RMSE": -ridge_scores["test_RMSE"],
    "R2": ridge_scores["test_R2"]
})

In [12]:
comparison = pd.DataFrame({
    "Mean baseline": baseline_folds[metrics].mean(),
    "Linear Regression": linear_folds[metrics].mean(),
    "Ridge": ridge_folds[metrics].mean()
}).T

display(comparison)
display(ridge_folds[metrics].agg(["mean", "std"]))

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Linear Regression,3.522677,4.607035,-0.092388
Ridge,3.490391,4.562138,-0.069352


,MAE,RMSE,R2
mean,3.490391,4.562138,-0.069352
std,0.421381,0.352943,0.233039


## Decision Tree Regression

A Decision Tree is evaluated to capture nonlinear relationships and
interactions between features.

Scaling is disabled because tree splits do not require comparable
feature scales. Original absences are retained, and nominal features
use the same reference-category encoding as the preceding models.

An initial maximum depth of 3 and minimum leaf size of 10 constrain
tree complexity.

In [13]:
from sklearn.tree import DecisionTreeRegressor

tree_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=False
)

tree_preprocessor.set_params(
    column_processing__nominal__drop="first"
)

tree_pipeline = Pipeline([
    ("preprocessing", tree_preprocessor),
    (
        "model",
        DecisionTreeRegressor(
            max_depth=3,
            min_samples_leaf=10,
            random_state=42
        )
    )
])

tree_scores = cross_validate(
    tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

tree_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -tree_scores["test_MAE"],
    "RMSE": -tree_scores["test_RMSE"],
    "R2": tree_scores["test_R2"]
})

In [14]:
comparison = pd.DataFrame({
    "Mean baseline": baseline_folds[metrics].mean(),
    "Linear Regression": linear_folds[metrics].mean(),
    "Ridge": ridge_folds[metrics].mean(),
    "Decision Tree": tree_folds[metrics].mean()
}).T

display(comparison)
display(tree_folds[metrics].agg(["mean", "std"]))

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Linear Regression,3.522677,4.607035,-0.092388
Ridge,3.490391,4.562138,-0.069352
Decision Tree,3.130386,4.176175,0.103267


,MAE,RMSE,R2
mean,3.130386,4.176175,0.103267
std,0.242535,0.307117,0.187839


## Random Forest Regression

Random Forest averages predictions from multiple decision trees trained
using resampled observations and randomness in feature selection.

It is evaluated to assess whether combining trees improves generalization
over the single Decision Tree. Scaling is disabled, and original
absences and reference-category encoding are retained.

In [15]:
from sklearn.ensemble import RandomForestRegressor

forest_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=False
)

forest_preprocessor.set_params(
    column_processing__nominal__drop="first"
)

forest_pipeline = Pipeline([
    ("preprocessing", forest_preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        )
    )
])

forest_scores = cross_validate(
    forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

forest_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -forest_scores["test_MAE"],
    "RMSE": -forest_scores["test_RMSE"],
    "R2": forest_scores["test_R2"]
})

In [16]:
comparison = pd.DataFrame({
    "Mean baseline": baseline_folds[metrics].mean(),
    "Linear Regression": linear_folds[metrics].mean(),
    "Ridge": ridge_folds[metrics].mean(),
    "Decision Tree": tree_folds[metrics].mean(),
    "Random Forest": forest_folds[metrics].mean()
}).T

display(comparison)
display(forest_folds[metrics].agg(["mean", "std"]))

,MAE,RMSE,R2
Mean baseline,3.402865,4.563458,-0.035023
Linear Regression,3.522677,4.607035,-0.092388
Ridge,3.490391,4.562138,-0.069352
Decision Tree,3.130386,4.176175,0.103267
Random Forest,3.001524,3.952507,0.205094


,MAE,RMSE,R2
mean,3.001524,3.952507,0.205094
std,0.367969,0.349360,0.133325


## Gradient Boosting Regression

Gradient Boosting combines small decision trees sequentially to improve
predictions by addressing errors from the existing ensemble.

Scaling is disabled. Original absences and reference-category encoding
are retained for consistency with the other initial comparisons.

In [17]:
from sklearn.ensemble import GradientBoostingRegressor

boosting_preprocessor = make_preprocessor(
    use_log_absences=False,
    scale_numeric=False
)

boosting_preprocessor.set_params(
    column_processing__nominal__drop="first"
)

boosting_pipeline = Pipeline([
    ("preprocessing", boosting_preprocessor),
    (
        "model",
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=2,
            min_samples_leaf=5,
            random_state=42
        )
    )
])

boosting_scores = cross_validate(
    boosting_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise"
)

boosting_folds = pd.DataFrame({
    "Fold": range(1, 6),
    "MAE": -boosting_scores["test_MAE"],
    "RMSE": -boosting_scores["test_RMSE"],
    "R2": boosting_scores["test_R2"]
})

In [18]:
comparison = pd.DataFrame({
    "Mean baseline": baseline_folds[metrics].mean(),
    "Linear Regression": linear_folds[metrics].mean(),
    "Ridge": ridge_folds[metrics].mean(),
    "Decision Tree": tree_folds[metrics].mean(),
    "Random Forest": forest_folds[metrics].mean(),
    "Gradient Boosting": boosting_folds[metrics].mean()
}).T

display(comparison.sort_values("MAE"))
display(boosting_folds[metrics].agg(["mean", "std"]))

,MAE,RMSE,R2
Random Forest,3.001524,3.952507,0.205094
Gradient Boosting,3.046055,3.945861,0.210174
Decision Tree,3.130386,4.176175,0.103267
Mean baseline,3.402865,4.563458,-0.035023
Ridge,3.490391,4.562138,-0.069352
Linear Regression,3.522677,4.607035,-0.092388


,MAE,RMSE,R2
mean,3.046055,3.945861,0.210174
std,0.409967,0.376844,0.118981


## Hyperparameter tuning

## Ridge Hyperparameter Tuning

MAE is the primary model-selection metric because it communicates
average absolute prediction error in grade points. RMSE and R²
are retained as supporting metrics.

Grid search compares Ridge penalty strengths and two absence
representations using the existing five-fold training cross-validation.

In [19]:
from sklearn.model_selection import GridSearchCV

ridge_preprocessing_options = []

for use_log in [False, True]:
    preprocessor = make_preprocessor(
        use_log_absences=use_log,
        scale_numeric=True
    )

    preprocessor.set_params(
        column_processing__nominal__drop="first"
    )

    ridge_preprocessing_options.append(preprocessor)

In [20]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid={
        "preprocessing": ridge_preprocessing_options,
        "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
    },
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    error_score="raise"
)

ridge_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...l', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.01, 0.1, ...], 'preprocessing': [Pipeline(step...sences'])]))]), Pipeline(step...es_log'])]))])]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'MAE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchange

In [21]:
best_index = ridge_search.best_index_
best_ridge = ridge_search.best_estimator_

selected_log = (
    best_ridge
    .named_steps["preprocessing"]
    .named_steps["absence_transform"]
    .kw_args["use_log"]
)

print("Selected alpha:", best_ridge.named_steps["model"].alpha)
print("Log absences:", selected_log)

ridge_tuned_summary = pd.DataFrame({
    "Mean": [
        -ridge_search.cv_results_["mean_test_MAE"][best_index],
        -ridge_search.cv_results_["mean_test_RMSE"][best_index],
        ridge_search.cv_results_["mean_test_R2"][best_index]
    ]
}, index=["MAE", "RMSE", "R2"])

ridge_tuned_summary

Selected alpha: 100
Log absences: True


,Mean
MAE,3.242256
RMSE,4.167685
R2,0.125253


## Random Forest Hyperparameter Tuning

In [22]:
forest_preprocessing_options = []

for use_log in [False, True]:
    preprocessor = make_preprocessor(
        use_log_absences=use_log,
        scale_numeric=False
    )

    preprocessor.set_params(
        column_processing__nominal__drop="first"
    )

    forest_preprocessing_options.append(preprocessor)

In [23]:
forest_tuning_pipeline = Pipeline([
    ("preprocessing", forest_preprocessing_options[0]),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=1
        )
    )
])

forest_search = GridSearchCV(
    estimator=forest_tuning_pipeline,
    param_grid={
        "preprocessing": forest_preprocessing_options,
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [2, 5, 10],
        "model__max_features": [0.8, 1.0]
    },
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    error_score="raise"
)

forest_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [None, 5, ...], 'model__max_features': [0.8, 1.0], 'model__min_samples_leaf': [2, 5, ...], 'preprocessing': [Pipeline(step...sences'])]))]), Pipeline(step...es_log'])]))])]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'MAE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for th

In [24]:
best_index = forest_search.best_index_
best_forest = forest_search.best_estimator_

print("Selected model settings:")
print({
    name: forest_search.best_params_[name]
    for name in [
        "model__max_depth",
        "model__min_samples_leaf",
        "model__max_features"
    ]
})

print(
    "Log absences:",
    best_forest.named_steps["preprocessing"]
    .named_steps["absence_transform"]
    .kw_args["use_log"]
)

forest_tuned_summary = pd.DataFrame({
    "Mean": [
        -forest_search.cv_results_["mean_test_MAE"][best_index],
        -forest_search.cv_results_["mean_test_RMSE"][best_index],
        forest_search.cv_results_["mean_test_R2"][best_index]
    ]
}, index=["MAE", "RMSE", "R2"])

forest_tuned_summary

Selected model settings:
{'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__max_features': 0.8}
Log absences: False


,Mean
MAE,2.964054
RMSE,3.936634
R2,0.208057


## Gradient Boosting Hyperparameter Tuning

In [25]:
boosting_preprocessing_options = []

for use_log in [False, True]:
    preprocessor = make_preprocessor(
        use_log_absences=use_log,
        scale_numeric=False
    )

    preprocessor.set_params(
        column_processing__nominal__drop="first"
    )

    boosting_preprocessing_options.append(preprocessor)

In [26]:
boosting_search = GridSearchCV(
    estimator=boosting_pipeline,
    param_grid={
        "preprocessing": boosting_preprocessing_options,
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.03, 0.1],
        "model__max_depth": [1, 2],
        "model__min_samples_leaf": [5, 10]
    },
    scoring=scoring,
    refit="MAE",
    cv=cv,
    n_jobs=-1,
    error_score="raise"
)

boosting_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__learning_rate': [0.03, 0.1], 'model__max_depth': [1, 2], 'model__min_samples_leaf': [5, 10], 'model__n_estimators': [100, 200], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'MAE': 'neg_mean_absolute_error', 'R2': 'r2', 'RMSE': 'neg_root_mean_squared_error'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'MAE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used

In [27]:
best_index = boosting_search.best_index_
best_boosting = boosting_search.best_estimator_

print("Selected model settings:")
print({
    name: value
    for name, value in boosting_search.best_params_.items()
    if name.startswith("model__")
})

print(
    "Log absences:",
    best_boosting.named_steps["preprocessing"]
    .named_steps["absence_transform"]
    .kw_args["use_log"]
)

boosting_tuned_summary = pd.DataFrame({
    "Mean": [
        -boosting_search.cv_results_["mean_test_MAE"][best_index],
        -boosting_search.cv_results_["mean_test_RMSE"][best_index],
        boosting_search.cv_results_["mean_test_R2"][best_index]
    ]
}, index=["MAE", "RMSE", "R2"])

boosting_tuned_summary

Selected model settings:
{'model__learning_rate': 0.03, 'model__max_depth': 2, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}
Log absences: False


,Mean
MAE,3.046962
RMSE,3.943664
R2,0.210082


## Comparison

In [28]:
tuned_comparison = pd.DataFrame({
    "Ridge": ridge_tuned_summary["Mean"],
    "Random Forest": forest_tuned_summary["Mean"],
    "Gradient Boosting": boosting_tuned_summary["Mean"]
}).T.sort_values("MAE")

tuned_comparison

,MAE,RMSE,R2
Random Forest,2.964054,3.936634,0.208057
Gradient Boosting,3.046962,3.943664,0.210082
Ridge,3.242256,4.167685,0.125253


## Model Selection

Select the tuned candidate with the lowest mean cross-validation MAE. These scores were used for selection and are not an independent final performance estimate. The selected search has already refitted its best pipeline on all training students.

In [29]:
tuned_searches = {
    "Ridge": ridge_search,
    "Random Forest": forest_search,
    "Gradient Boosting": boosting_search
}

selected_model_name = tuned_comparison["MAE"].idxmin()
selected_search = tuned_searches[selected_model_name]
final_model = selected_search.best_estimator_

print("Selected model:", selected_model_name)
print("Training students:", len(X_train))
print("Held-out test students:", len(X_test))
print("Test performance has not been evaluated.")

Selected model: Random Forest
Training students: 316
Held-out test students: 79
Test performance has not been evaluated.
